<a href="https://colab.research.google.com/github/LinckerFrank/FINALE-PROYECT-DLSA/blob/main/FINAL_PROYECT_DLSD_Primer_Avance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Descargar directamente el archivo de Excel desde tu GitHub al Colab
url = "https://raw.githubusercontent.com/TU_USUARIO/NOMBRE_DEL_REPO/main/tickets_historico.xlsx"
# *Reemplaza TU_USUARIO y NOMBRE_DEL_REPO con los tuyos reales

# Si prefieres generarlo al instante en el Colab para probar:
import numpy as np
fechas = pd.date_range(start='2026-01-01', end='2026-03-31', freq='h')
df = pd.DataFrame({
    'fecha_hora': fechas,
    'volumen_tickets': np.random.poisson(lam=5, size=len(fechas)),
    'tiempo_atencion_promedio': np.random.uniform(3, 15, size=len(fechas))
})

print("¡Datos cargados correctamente en Google Colab!")
print(df.head())

¡Datos cargados correctamente en Google Colab!
           fecha_hora  volumen_tickets  tiempo_atencion_promedio
0 2026-01-01 00:00:00                4                 14.023796
1 2026-01-01 01:00:00                4                 12.645196
2 2026-01-01 02:00:00                4                  8.555533
3 2026-01-01 03:00:00                5                  3.822763
4 2026-01-01 04:00:00                4                 13.413288


In [2]:
# 1. Asegurarnos de que la fecha_hora sea reconocida como formato de fecha
df['fecha_hora'] = pd.to_datetime(df['fecha_hora'])

# 2. Extraer características temporales clave (Ingeniería de Características)
df['dia_semana_num'] = df['fecha_hora'].dt.dayofweek  # Lunes=0, Domingo=6
df['hora_dia'] = df['fecha_hora'].dt.hour
df['es_fin_de_semana'] = df['dia_semana_num'].apply(lambda x: 1 if x >= 5 else 0)

# 3. Ver cómo quedan las nuevas columnas procesadas
print("¡Características temporales extraídas con éxito!")
print(df[['fecha_hora', 'volumen_tickets', 'hora_dia', 'es_fin_de_semana']].head())

¡Características temporales extraídas con éxito!
           fecha_hora  volumen_tickets  hora_dia  es_fin_de_semana
0 2026-01-01 00:00:00                4         0                 0
1 2026-01-01 01:00:00                4         1                 0
2 2026-01-01 02:00:00                4         2                 0
3 2026-01-01 03:00:00                5         3                 0
4 2026-01-01 04:00:00                4         4                 0


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

# 1. Definir nuestras Variables de Entrada (X) y lo que queremos Predecir (y)
# Usaremos la hora del día, el día de la semana y si es fin de semana para predecir el volumen de tickets
X = df[['hora_dia', 'dia_semana_num', 'es_fin_de_semana']]
y = df['volumen_tickets']

# 2. Dividir los datos: 80% para entrenar al modelo y 20% para evaluarlo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Construir la Red Neuronal (Deep Learning simple de capas densas)
modelo = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)), # Capa oculta 1
    Dense(8, activation='relu'),                                  # Capa oculta 2
    Dense(1)                                                      # Capa de salida (el volumen predicho)
])

# 4. Configurar el modelo
modelo.compile(optimizer='adam', loss='mean_squared_error')

# 5. Entrenar el modelo (aquí es donde la red "aprende" de los datos)
print("Entrenando el modelo de Deep Learning...")
historial = modelo.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
print("¡Modelo entrenado con éxito!")

# 6. Probar una predicción rápida
predicciones = modelo.predict(X_test)
print("Primeras 5 predicciones de volumen de tickets vs valor real:")
for i in range(5):
    print(f"Predicho: {predicciones[i][0]:.2f} | Real: {y_test.iloc[i]}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Entrenando el modelo de Deep Learning...
¡Modelo entrenado con éxito!
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Primeras 5 predicciones de volumen de tickets vs valor real:
Predicho: 5.30 | Real: 7
Predicho: 4.73 | Real: 8
Predicho: 3.41 | Real: 2
Predicho: 3.98 | Real: 4
Predicho: 5.27 | Real: 3


In [4]:
import numpy as np

# 1. Tomamos las predicciones que hizo nuestro modelo de Deep Learning
# (Convertimos los resultados de la red neuronal a una lista plana)
volumen_predicho = predicciones.flatten()

# 2. Definimos la regla del negocio basada en tu KPI
# Supongamos que un asesor puede atender de forma óptima un promedio de 6 tickets por hora
# para garantizar que el tiempo de atención se mantenga por debajo de los 10 minutos.
CAPACIDAD_MAX_ASESOR_POR_HORA = 6

# 3. Calculamos la cantidad de asesores necesarios por cada hora predicha
# Usamos np.ceil para redondear siempre hacia arriba (no podemos tener medio asesor)
asesores_necesarios = np.ceil(volumen_predicho / CAPACIDAD_MAX_ASESOR_POR_HORA)

# 4. Mostramos el resultado de la planificación para la próxima semana/mes
print("=== RECOMENDACIÓN DE PERSONAL PARA CUMPLIR EL KPI (<= 10 MIN) ===")
for i in range(5):
    print(f"Registro {i+1} -> Tickets previstos: {volumen_predicho[i]:.1f} | Asesores mínimos requeridos: {int(asesores_necesarios[i])} operadores")

=== RECOMENDACIÓN DE PERSONAL PARA CUMPLIR EL KPI (<= 10 MIN) ===
Registro 1 -> Tickets previstos: 5.3 | Asesores mínimos requeridos: 1 operadores
Registro 2 -> Tickets previstos: 4.7 | Asesores mínimos requeridos: 1 operadores
Registro 3 -> Tickets previstos: 3.4 | Asesores mínimos requeridos: 1 operadores
Registro 4 -> Tickets previstos: 4.0 | Asesores mínimos requeridos: 1 operadores
Registro 5 -> Tickets previstos: 5.3 | Asesores mínimos requeridos: 1 operadores
